# Universite de Haute-Alsace - Faculte des Sciences et Techniques
## M2 - IM - IMDS - MIAGE

# Apprentissage statistique et automatique
## Devoir 1 : Introduction et apprentissage

**Objectif.** Ce notebook compare experimentalement le cadre PAC realisable et le cadre PAC agnostique pour des classifieurs a seuil sur $X=[0,1]$. La regle d'apprentissage est un algorithme de minimisation du risque empirique (ERM), implemente sans bibliotheque d'apprentissage automatique.

## 1. Protocole experimental

Nous utilisons un seuil cible $\\theta=0.60$, les tailles d'echantillon $m\\in\\{5,10,20,50,100,200\\}$, un niveau de precision $\\varepsilon=0.10$ et un niveau de confiance $\\delta=0.05$. Chaque experience est repetee 500 fois. Dans l'experience PAC agnostique, chaque etiquette observee est independamment inversee avec une probabilite $\\eta=0.10$.

Pour chaque ensemble d'apprentissage, le risque empirique est mesure sur les donnees d'apprentissage. Le vrai risque est approche par l'erreur sur un ensemble de test independant de 20 000 points. Comme les etiquettes de test sont generees selon le meme modele que les etiquettes d'apprentissage, cette erreur estime le risque de population pertinent.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Use a standard output function so the notebook also runs in minimal Python environments.
display = print

SEED = 2026
rng = np.random.default_rng(SEED)
THETA = 0.60
EPSILON = 0.10
DELTA = 0.05
ETA = 0.10
SAMPLE_SIZES = np.array([5, 10, 20, 50, 100, 200])
N_REPETITIONS = 500
N_TEST = 20_000
plt.style.use('seaborn-v0_8-whitegrid')

## 2. Generation des donnees

Le classifieur cible est $f_\\theta(x)=1$ lorsque $x>\\theta$ et $0$ sinon. La fonction ci-dessous genere des entrees uniformes et peut inverser les etiquettes selon un taux de bruit donne.

In [ ]:
def target_function(x, theta=THETA):
    return (np.asarray(x) > theta).astype(int)


def generate_dataset(m, theta=THETA, noise_rate=0.0, random_generator=rng):
    x = random_generator.uniform(0.0, 1.0, size=m)
    y = target_function(x, theta)
    if noise_rate > 0.0:
        flips = random_generator.random(m) < noise_rate
        y = np.where(flips, 1 - y, y)
    order = np.argsort(x)
    return x[order], y[order]


x_example, y_example = generate_dataset(50)
example_table = pd.DataFrame({'x': x_example, 'y': y_example})
display(example_table.head(10))

fig, ax = plt.subplots(figsize=(9, 2.8))
ax.scatter(x_example, y_example, c=y_example, cmap='coolwarm', s=42, edgecolor='black', linewidth=0.3)
ax.axvline(THETA, color='black', linestyle='--', label=fr'target $\theta={THETA}$')
ax.set(xlim=(0, 1), ylim=(-0.15, 1.15), xlabel='Input x', ylabel='Label y', title='One realizable training dataset')
ax.set_yticks([0, 1])
ax.legend()
plt.show()

## 3. ERM pour les classifieurs a seuil

Un seuil $t$ predit $0$ a gauche de $t$ et $1$ a droite. Pour un echantillon trie, les seules valeurs de seuil qui peuvent modifier les predictions sont les intervalles avant le premier point, entre deux points consecutifs et apres le dernier point. Nous evaluons tous ces seuils candidats et choisissons celui qui minimise la perte empirique 0-1. En cas d'egalite, le plus petit seuil est choisi afin de rendre le resultat deterministe.

In [ ]:
def threshold_predict(x, threshold):
    return (np.asarray(x) > threshold).astype(int)


def erm_threshold(x, y):
    x = np.asarray(x)
    y = np.asarray(y).astype(int)
    order = np.argsort(x)
    x_sorted, y_sorted = x[order], y[order]
    candidates = np.concatenate(([0.0], (x_sorted[:-1] + x_sorted[1:]) / 2, [1.0]))
    errors = np.array([np.mean(threshold_predict(x_sorted, t) != y_sorted) for t in candidates])
    best_index = np.flatnonzero(errors == errors.min())[0]
    return float(candidates[best_index]), float(errors[best_index])


threshold_hat, empirical_error = erm_threshold(x_example, y_example)
print(f'Estimated threshold: {threshold_hat:.3f}')
print(f'Empirical error: {empirical_error:.3f}')

## 4. Risque empirique et vrai risque

La fonction suivante entraine l'ERM sur un echantillon puis l'evalue sur un echantillon de test independant. Dans le cadre realisable, l'ERM trouve generalement un seuil avec une erreur d'apprentissage nulle, mais son erreur de test peut rester positive pour les petits echantillons car le seuil estime n'est pas exactement le seuil cible.

In [ ]:
def evaluate_one_run(m, theta=THETA, noise_rate=0.0, n_test=N_TEST, random_generator=rng):
    x_train, y_train = generate_dataset(m, theta, noise_rate, random_generator)
    threshold, empirical_error = erm_threshold(x_train, y_train)
    x_test, y_test = generate_dataset(n_test, theta, noise_rate, random_generator)
    true_error = float(np.mean(threshold_predict(x_test, threshold) != y_test))
    return threshold, empirical_error, true_error


risk_rows = []
for m in SAMPLE_SIZES:
    threshold, train_error, test_error = evaluate_one_run(m)
    risk_rows.append({'m': m, 'estimated_threshold': threshold, 'empirical_error': train_error, 'true_error': test_error})
risk_table = pd.DataFrame(risk_rows)
display(risk_table.round(4))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(risk_table['m'], risk_table['empirical_error'], 'o-', label='Empirical error')
ax.plot(risk_table['m'], risk_table['true_error'], 's-', label='Independent test error')
ax.set(xlabel='Training size m', ylabel='Error', title='One-run comparison of empirical and true risk')
ax.set_xscale('log')
ax.set_xticks(SAMPLE_SIZES, labels=SAMPLE_SIZES)
ax.legend()
plt.show()

## 5. Verification experimentale de l'apprentissage PAC

Pour chaque $m$, nous repetons l'experience realisable 500 fois et estimons $\\Pr(L(h)>\\varepsilon)$ par la proportion d'erreurs de test independantes superieures a $\\varepsilon$. La theorie PAC predit que cette probabilite d'echec diminue lorsque $m$ augmente. La borne theorique $2e^{-2m\\varepsilon^2}$ est ajoutee comme reference : il s'agit d'une borne superieure generale, souvent plus grande que la probabilite mesuree.

In [ ]:
def repeated_experiment(sample_sizes, n_repetitions, theta, noise_rate, epsilon, n_test, random_generator):
    rows = []
    for m in sample_sizes:
        empirical_errors = []
        true_errors = []
        thresholds = []
        for _ in range(n_repetitions):
            threshold, train_error, test_error = evaluate_one_run(m, theta, noise_rate, n_test, random_generator)
            thresholds.append(threshold)
            empirical_errors.append(train_error)
            true_errors.append(test_error)
        empirical_errors = np.array(empirical_errors)
        true_errors = np.array(true_errors)
        rows.append({
            'm': m,
            'mean_empirical_error': empirical_errors.mean(),
            'mean_true_error': true_errors.mean(),
            'std_true_error': true_errors.std(ddof=1),
            'failure_probability': np.mean(true_errors > epsilon),
            'mean_threshold': np.mean(thresholds)
        })
    return pd.DataFrame(rows)


pac_results = repeated_experiment(SAMPLE_SIZES, N_REPETITIONS, THETA, 0.0, EPSILON, N_TEST, rng)
pac_results['pac_reference_bound'] = np.minimum(1.0, 2 * np.exp(-2 * pac_results['m'] * EPSILON**2))
display(pac_results.round(4))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(pac_results['m'], pac_results['mean_empirical_error'], 'o-', label='Mean empirical error')
axes[0].plot(pac_results['m'], pac_results['mean_true_error'], 's-', label='Mean true error')
axes[0].set(xlabel='Training size m', ylabel='Error', title='Realizable setting: average risks')
axes[0].set_xscale('log'); axes[0].set_xticks(SAMPLE_SIZES, labels=SAMPLE_SIZES); axes[0].legend()
axes[1].plot(pac_results['m'], pac_results['failure_probability'], 'o-', label=r'Observed $P(L(h)>\varepsilon)$')
axes[1].plot(pac_results['m'], pac_results['pac_reference_bound'], '--', label='Reference bound')
axes[1].axhline(DELTA, color='black', linestyle=':', label=fr'$\delta={DELTA}$')
axes[1].set(xlabel='Training size m', ylabel='Probability', title=fr'PAC failure probability ($\varepsilon={EPSILON}$)')
axes[1].set_xscale('log'); axes[1].set_xticks(SAMPLE_SIZES, labels=SAMPLE_SIZES); axes[1].set_ylim(-0.02, 1.02); axes[1].legend()
plt.tight_layout(); plt.show()

## 6. Du PAC au PAC agnostique

Nous fixons maintenant $\\eta=0.10$. Les etiquettes observees ne sont plus parfaitement compatibles avec un seuil : meme le meilleur seuil peut donc commettre des erreurs d'apprentissage. L'ERM minimise l'erreur empirique, mais l'optimum empirique est influence par les inversions aleatoires. Davantage de donnees reduit l'erreur d'estimation et devrait ameliorer le seuil appris et le risque de population, meme si le bruit irreductible demeure.

In [ ]:
agnostic_results = repeated_experiment(SAMPLE_SIZES, N_REPETITIONS, THETA, ETA, EPSILON, N_TEST, rng)
display(agnostic_results.round(4))

comparison = pac_results[['m', 'mean_empirical_error', 'mean_true_error', 'failure_probability']].merge(
    agnostic_results[['m', 'mean_empirical_error', 'mean_true_error', 'failure_probability']],
    on='m', suffixes=('_realizable', '_agnostic')
)
display(comparison.round(4))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for label, results, color in [('Realizable', pac_results, 'tab:blue'), ('Agnostic', agnostic_results, 'tab:orange')]:
    axes[0].plot(results['m'], results['mean_empirical_error'], 'o-', color=color, label=f'{label}: empirical')
    axes[0].plot(results['m'], results['mean_true_error'], 's--', color=color, label=f'{label}: true')
axes[0].set(xlabel='Training size m', ylabel='Error', title='Realizable versus noisy labels')
axes[0].set_xscale('log'); axes[0].set_xticks(SAMPLE_SIZES, labels=SAMPLE_SIZES); axes[0].legend(ncol=2)
axes[1].plot(pac_results['m'], pac_results['failure_probability'], 'o-', label='Realizable')
axes[1].plot(agnostic_results['m'], agnostic_results['failure_probability'], 's-', label='Agnostic, eta=0.10')
axes[1].axhline(DELTA, color='black', linestyle=':', label=fr'delta={DELTA}')
axes[1].set(xlabel='Training size m', ylabel=fr'P(L(h)>\varepsilon), epsilon={EPSILON}', title='Generalization failure probability')
axes[1].set_xscale('log'); axes[1].set_xticks(SAMPLE_SIZES, labels=SAMPLE_SIZES); axes[1].set_ylim(-0.02, 1.02); axes[1].legend()
plt.tight_layout(); plt.show()

print(f'Agnostic setting: expected irreducible test noise level is approximately eta={ETA:.2f}.')
print(f'At m={SAMPLE_SIZES[-1]}, the mean empirical error is {agnostic_results.iloc[-1].mean_empirical_error:.3f} and the mean true error is {agnostic_results.iloc[-1].mean_true_error:.3f}.')

## 7. Conclusion

Dans le cadre PAC realisable, les etiquettes sont generees exactement par un seuil. L'ERM peut donc atteindre une erreur empirique nulle, tandis que son vrai risque reste positif pour les petits echantillons car le seuil appris peut etre eloigne de $\\theta$. Lorsque $m$ augmente, le seuil appris se concentre autour du seuil cible, le vrai risque diminue et la probabilite estimee $\\Pr(L(h)>\\varepsilon)$ baisse generalement.

Dans le cadre PAC agnostique, les inversions d'etiquettes creent des contradictions qu'aucun seuil ne peut supprimer completement. L'erreur empirique ne reste donc generalement pas nulle. Augmenter $m$ ameliore tout de meme la generalisation en reduisant la variabilite d'estimation, mais le risque de population ne peut pas descendre sous le plancher impose par le bruit $\\eta$. Les risques empirique et reel restent ainsi non nuls, et la convergence se fait vers le meilleur seuil pour la distribution bruitee plutot que vers un classifieur parfaitement realisable.

Ces observations illustrent la distinction centrale : l'apprentissage PAC exploite une cible realisable et peut atteindre une erreur arbitrairement petite avec suffisamment de donnees, tandis que l'apprentissage PAC agnostique autorise un desaccord ou un bruit inevitable et demande a l'ERM de se rapprocher de la meilleure hypothese disponible dans la classe. Les probabilites d'echec numeriques sont estimees sur 500 repetitions ; de petites fluctuations entre tailles d'echantillon voisines sont donc normales.